# Análise individual: Plano Brasileiro de Inteligência Artificial (PBIA)

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 (análise individual), conforme a skill 02 ([`skill_análise_individual.md`](skill_análise_individual.md))
**Insumo:** `PBIA_2025_extracao.json`, produzido pela skill 01 ([`skill_extração.md`](skill_extração.md))

---

> ⚠️ **Visualizações prévias.** Todas as visualizações geradas neste notebook (gráficos e tabelas) são **prévias** e compõem **apenas uma base para o início da construção de novos gráficos**. As novas visualizações serão construídas de acordo com os comandos e as exigências do pesquisador, que são obedecidos em última instância. Nenhuma visualização, tabela ou relatório vai para a pasta `resultados/` sem comando expresso do pesquisador.

## 1. Do que se trata

Este notebook analisa, de forma **individual**, o texto do plano *AI for the Good of All: Brazilian Artificial Intelligence Plan* (PBIA). Ele faz parte do fluxo de análise da pasta *Planos Nacionais de IA*: a skill 01 extrai o texto essencial de cada plano; esta etapa (skill 02) analisa cada plano isoladamente; a skill 03 compara os planos entre si.

## 2. Finalidade

A finalidade deste notebook é **gerar visualizações e interpretações sobre o plano analisado**, por meio de gráficos, tabelas e medidas textuais. As análises permitem conhecer:

- a **extensão** e a estrutura do texto;
- os **termos mais comuns** e como se distribuem ao longo do documento;
- as **expressões recorrentes**.

## 3. Insumo e recorte do conteúdo

São utilizados **apenas os resultados da skill 01**, ou seja, o arquivo JSON do plano gerado por aquela etapa. O relatório de extração (`relatorios_extracao/`) é apenas informativo e **não** é consultado. Do JSON, interessam **somente**:

- o **título** e o **subtítulo** do plano (`document.title` e `document.subtitle`), que o identificam e, por padrão, **entram na contagem** como unidade 0 (`INCLUIR_TITULO_NA_CONTAGEM = True`), porque fazem parte do texto essencial do plano;
- o **conteúdo integral e literal de cada unidade** (`sections[].text`): títulos internos, parágrafos e itens de lista. É o conteúdo analisado;
- o **tipo** (`sections[].type`) e a **categoria estrutural** (`sections[].category`) de cada unidade, usados **apenas como filtros**. Por padrão, nenhum filtro retira conteúdo: os títulos internos entram (`INCLUIR_TITULOS_INTERNOS = True`) e todas as categorias entram (`CATEGORIAS_EXCLUIDAS = []`). Retirar a apresentação (`Foreword`) ou os anexos (`Annex`) é decisão do pesquisador.

O identificador de cada unidade (`id`) é usado apenas como índice de posição no documento. **Todo conteúdo periférico fica fora da análise**: numeração (`number`), nível (`level`), página (`page`), caminho estrutural (`path`) e os demais metadados (`issuer`, `date`, `period` etc.). A função `carregar_extracao` descarta esse conteúdo no momento da leitura.

## 4. Método (visão geral)

| Etapa | Procedimento |
|---|---|
| Verificação | Integridade do JSON, origem na skill 01 e no corpus *Planos Nacionais de IA*, idioma `en` |
| Recorte | Título e subtítulo como unidade 0; filtros por tipo, por categoria e por unidade, definidos na configuração e informados no perfil |
| Pré-processamento | spaCy (`en_core_web_sm`): tokenização, reunião de compostos com hífen, lematização, classes gramaticais, *stopwords* |
| Exame de inflação | Busca de artefatos que multipliquem termos (textos idênticos em várias unidades, rótulos numerados no início dos títulos internos). Tratamento apenas por parâmetros visíveis |
| Frequências | Frequência absoluta, por 1.000 tokens, alcance (unidades em que o termo aparece) e dispersão (DP normalizado de Gries) |
| Expressões | Bigramas de tokens de conteúdo adjacentes no texto original |

O detalhamento de cada procedimento está na configuração, nas funções comentadas e no texto que acompanha cada visualização.

## 5. Transparência das escolhas

**Todo método utilizado e todas as escolhas tomadas pelo pesquisador estão presentes e verificáveis** em dois lugares:

1. **no código**: os parâmetros ficam reunidos na célula de configuração, e cada função traz comentários sobre o que faz;
2. **na descrição que segue cada visualização**, composta por: *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

As tabelas completas (e não apenas os recortes exibidos nos gráficos), as figuras e os demais arquivos **só são exportados para a pasta `resultados/` por comando expresso do pesquisador** (skill 02, seção 4.4). Sem esse comando, o notebook apenas exibe os resultados.

## 6. Princípios

Todo o conteúdo deste notebook segue os princípios da **skill 02** (análise individual dos planos). Por isso, **não incorpora análise conjunta**: nenhum resultado aqui é comparado com outro plano. A comparação entre planos é feita em notebook próprio, segundo a skill 03.

> As interpretações são redigidas **somente após a execução** do notebook, com base nos valores efetivamente obtidos. Se o JSON, os filtros ou os parâmetros mudarem, os números mudam e as interpretações precisam ser revistas.

## 7. Configuração

Todos os parâmetros editáveis estão nesta célula. Qualquer alteração deve ser registrada. A gravação em `resultados/` está desativada (`SALVAR_SAIDAS = False`) e só é ativada por comando do pesquisador.

In [ ]:
import hashlib
import json
import math
import platform
import re
import unicodedata
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter, MaxNLocator

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 140)

# =====================================================================
# CONFIGURAÇÃO — parâmetros editáveis pelo pesquisador
# =====================================================================
SIGLA = "PBIA_2025"
ARQUIVO_JSON = "PBIA_2025_extracao.json"       # entregável da skill 01 (um único JSON por plano)

# Saídas (figuras, tabelas, registro). Resultados anteriores nunca são sobrescritos:
# para uma nova execução com mudanças, altere VERSAO_SAIDA (v2, v3, ...).
NOME_SAIDA = SIGLA
VERSAO_SAIDA = "v1"
# Gravação em resultados/ DESATIVADA: visualizações, tabelas e relatórios só vão para
# resultados/ por comando expresso do pesquisador (skill 02, seção 4.4).
SALVAR_SAIDAS = False
SOBRESCREVER = False

# Recorte do conteúdo (skill 02, seção 3.1). Nenhum filtro altera o texto das unidades.
INCLUIR_TITULO_NA_CONTAGEM = True   # título e subtítulo entram na contagem como unidade 0
INCLUIR_TITULOS_INTERNOS = True     # unidades type = "heading" (títulos internos)
CATEGORIAS_EXCLUIDAS = []           # ex.: ["Foreword"] ou ["Annex"] (decisão do pesquisador)
UNIDADES_EXCLUIDAS = {}             # {id: "motivo"}: tratamento de inflação (skill 02, seção 3.5)

# Pré-processamento
IDIOMA_ESPERADO = "en"
MODELO_SPACY = "en_core_web_sm"
CLASSES_DE_CONTEUDO = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []    # lemas a remover além da lista do spaCy (decisão do pesquisador)
STOPWORDS_MANTIDAS = []      # lemas da lista do spaCy que devem ser mantidos (decisão do pesquisador)
CORRECOES_DE_LEMA = {
    "datum": "data",         # o spaCy lematiza "data" como "datum"; mantém-se "data", forma usada nos textos
}

# Recortes de visualização (as tabelas exportadas são sempre completas)
TOP_N_LEMAS = 25
TOP_N_DISPERSAO = 15
TOP_N_BIGRAMAS = 20
FREQ_MINIMA_BIGRAMA = 2

# Incerteza (bootstrap por unidades). Reservado para as análises com intervalo de confiança,
# que entram por comando do pesquisador (skill 02, seção 4.2); não é usado nas visualizações-padrão.
N_BOOTSTRAP = 2000
NIVEL_CONFIANCA = 0.95
SEMENTE = 2026

## 8. Ambiente, estilo e funções

Restrição de pasta, pasta de saída (usada apenas quando o pesquisador ordenar a gravação em `resultados/`), padrão gráfico e funções auxiliares.

In [ ]:
# ---------------------------------------------------------------------------
# Restrição de ambiente: o notebook só roda de dentro da pasta "Planos Nacionais de IA".
# ---------------------------------------------------------------------------
PASTA = Path.cwd()
if unicodedata.normalize("NFC", PASTA.name) != "Planos Nacionais de IA":
    raise RuntimeError(
        "Este notebook deve ser executado de dentro da pasta 'Planos Nacionais de IA' "
        f"(restrição de ambiente da skill). Pasta atual: {PASTA}"
    )

# ---------------------------------------------------------------------------
# Pasta de saída versionada: resultados anteriores nunca são sobrescritos.
# ---------------------------------------------------------------------------
DIR_SAIDA = PASTA / "resultados" / NOME_SAIDA / VERSAO_SAIDA
if SALVAR_SAIDAS:
    if DIR_SAIDA.exists() and any(DIR_SAIDA.iterdir()) and not SOBRESCREVER:
        raise FileExistsError(
            f"A pasta de saída '{DIR_SAIDA.relative_to(PASTA)}' já contém arquivos. "
            "Para preservar a versão anterior, altere VERSAO_SAIDA (ex.: 'v2') na configuração. "
            "Use SOBRESCREVER = True somente se a substituição for intencional."
        )
    DIR_SAIDA.mkdir(parents=True, exist_ok=True)
SAIDAS_GRAVADAS = []   # arquivos gravados nesta execução (vão para o registro de execução)

# ---------------------------------------------------------------------------
# Estilo das figuras (padrão acadêmico: fonte >= 9 pt, 300 dpi, PNG + PDF)
# ---------------------------------------------------------------------------
# Cor fixa por plano, igual em todos os notebooks da pasta (skill 02, seção 3.7).
# Paleta inicial, editável pelo pesquisador; planos do mesmo país ou bloco recebem tons
# da mesma cor. Como verde e vermelho se confundem para leitores com daltonismo, os gráficos
# com mais de um plano identificam cada um também por rótulo direto ou marcador.
COR_DOCUMENTO = {
    "PBIA_2025": "#2E9E4F",    # Brasil (verde)
    "EUA_2025": "#1F3F8F",     # Estados Unidos (azul-escuro)
    "UE_2025": "#EDB120",      # União Europeia (amarelo)
    "China_2025": "#B2182B",   # China (vermelho)
}
COR_TINTA = "#0b0b0b"     # texto principal
COR_TINTA_2 = "#52514e"   # texto secundário (rótulos, anotações)
COR_GRADE = "#e4e3df"     # grade e linhas de apoio

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": COR_TINTA_2,
    "axes.linewidth": 0.6,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": COR_GRADE,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "text.color": COR_TINTA,
    "axes.labelcolor": COR_TINTA,
    "xtick.color": COR_TINTA_2,
    "ytick.color": COR_TINTA_2,
    "legend.frameon": False,
    "pdf.fonttype": 42,
})

# ---------------------------------------------------------------------------
# Funções auxiliares de formatação e gravação
# ---------------------------------------------------------------------------
def num(x, casas=2):
    """Formata número no padrão brasileiro (vírgula decimal, ponto de milhar)."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "—"
    texto = f"{x:,.{casas}f}"
    return texto.replace(",", "§").replace(".", ",").replace("§", ".")


# Rótulos de eixo com vírgula decimal.
FMT_BR = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))


def exibir(df, casas=2):
    """Exibe uma tabela com vírgula decimal, sem alterar os valores."""
    display(df.style.format(precision=casas, decimal=",", thousands=".", na_rep="—"))


def salvar_figura(fig, nome):
    """Grava a figura em PNG (300 dpi) e PDF na pasta de saída versionada."""
    if SALVAR_SAIDAS:
        for ext in ("png", "pdf"):
            fig.savefig(DIR_SAIDA / f"{nome}.{ext}")
            SAIDAS_GRAVADAS.append(f"{nome}.{ext}")


def salvar_tabela(df, nome, index=True):
    """Grava a tabela completa em CSV (UTF-8 com BOM, ';', vírgula decimal)."""
    if SALVAR_SAIDAS:
        df.to_csv(DIR_SAIDA / f"{nome}.csv", sep=";", decimal=",", encoding="utf-8-sig", index=index)
        SAIDAS_GRAVADAS.append(f"{nome}.csv")


if SIGLA not in COR_DOCUMENTO:
    raise KeyError(f"Defina a cor do plano {SIGLA!r} em COR_DOCUMENTO (skill 02, seção 3.7).")
COR = COR_DOCUMENTO[SIGLA]

print(f"Pasta de trabalho: {PASTA}")
print(f"Saídas: {DIR_SAIDA.relative_to(PASTA) if SALVAR_SAIDAS else 'gravação desativada'}")

In [ ]:
# ===========================================================================
# PIPELINE DE PRÉ-PROCESSAMENTO E MEDIDAS
# (idêntico em todos os notebooks de análise da pasta "Planos Nacionais de IA")
# ===========================================================================

# Token (palavra) = sequência de letras, admitindo hífen interno (multi-stakeholder)
# ou sigla com pontos (U.S.). Números, pontuação e símbolos NÃO são tokens.
RE_PALAVRA = re.compile(r"(?:[^\W\d_]\.){2,}|[^\W\d_]+(?:[-‐‑][^\W\d_]+)*")
HIFENS = {"-", "‐", "‑"}

# Valores admitidos nos campos usados como filtro (skill 01, seção 4.3).
TIPOS_VALIDOS = {"heading", "paragraph", "list_item"}
CATEGORIAS_VALIDAS = {"Foreword", "Executive summary", "Body", "Box", "Annex"}
ROTULOS_TIPO = {
    "title": "Título e subtítulo (unidade 0)",
    "heading": "Títulos internos",
    "paragraph": "Parágrafos",
    "list_item": "Itens de lista",
}


def normalizar_hifen(texto):
    """Uniformiza variantes tipográficas do hífen (apenas nas chaves de contagem;
    o texto do JSON não é alterado)."""
    return texto.replace("‐", "-").replace("‑", "-")


def carregar_extracao(caminho):
    """Lê o JSON produzido pela skill 01 e devolve SOMENTE o título, o subtítulo e as
    unidades (id, tipo, categoria e texto integral). Todo o conteúdo periférico (number,
    level, page, path e demais metadados) é descartado aqui.
    Interrompe a execução se o arquivo não passar nas verificações de integridade."""
    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo '{caminho.name}' não encontrado na pasta. "
            "Execute antes a skill 01 (skill_extração.md) para gerar o JSON do plano."
        )
    bruto = caminho.read_bytes()
    dados = json.loads(bruto.decode("utf-8"))

    problemas = [f"chave '{c}' ausente" for c in ("document", "extraction", "sections") if c not in dados]
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    extracao, documento, secoes = dados["extraction"], dados["document"], dados["sections"]
    if extracao.get("skill") != "skill_extração":
        problemas.append("o JSON não registra ter sido produzido pela skill 01 (extraction.skill)")
    if extracao.get("corpus") != "Planos Nacionais de IA":
        problemas.append("o JSON não registra o corpus 'Planos Nacionais de IA' (extraction.corpus)")
    if [s.get("id") for s in secoes] != list(range(1, len(secoes) + 1)):
        problemas.append("os ids das unidades não são sequenciais a partir de 1")
    vazias = [s.get("id") for s in secoes if not isinstance(s.get("text"), str) or not s["text"].strip()]
    if vazias:
        problemas.append(f"unidades com texto vazio: {vazias}")
    if extracao.get("units_extracted") != len(secoes):
        problemas.append(
            f"extraction.units_extracted = {extracao.get('units_extracted')}, mas há {len(secoes)} unidades"
        )
    tipos = {s.get("type") for s in secoes} - TIPOS_VALIDOS
    if tipos:
        problemas.append(f"tipos de unidade não previstos pela skill 01: {sorted(map(str, tipos))}")
    categorias = {s.get("category") for s in secoes} - CATEGORIAS_VALIDAS
    if categorias:
        problemas.append(f"categorias não previstas pela skill 01: {sorted(map(str, categorias))}")
    if not documento.get("title"):
        problemas.append("título (document.title) ausente")
    if documento.get("language") != IDIOMA_ESPERADO:
        problemas.append(
            f"idioma declarado no JSON: {documento.get('language')!r}; esperado: {IDIOMA_ESPERADO!r}"
        )
    if problemas:
        raise ValueError("Falha na verificação do JSON:\n- " + "\n- ".join(problemas))

    return {
        "arquivo": caminho.name,
        "sha256": hashlib.sha256(bruto).hexdigest(),
        "titulo": documento["title"],
        "subtitulo": documento.get("subtitle"),
        "idioma": documento.get("language"),
        # SOMENTE id (posição), tipo e categoria (filtros) e texto (conteúdo analisado)
        "unidades": [
            {"id": s["id"], "tipo": s["type"], "categoria": s["category"], "texto": s["text"]}
            for s in secoes
        ],
    }


def aplicar_recorte(documento):
    """Monta as unidades analisadas a partir dos parâmetros de recorte da configuração e
    registra as unidades retiradas por cada filtro. Cada unidade retirada é atribuída ao
    primeiro filtro que a retira, nesta ordem: título e subtítulo; títulos internos;
    categorias; unidades excluídas. Nenhum texto é alterado."""
    ids_validos = {u["id"] for u in documento["unidades"]}
    inexistentes = sorted(set(UNIDADES_EXCLUIDAS) - ids_validos)
    if inexistentes:
        raise ValueError(f"UNIDADES_EXCLUIDAS contém ids que não existem no JSON: {inexistentes}")
    sem_motivo = [i for i, motivo in UNIDADES_EXCLUIDAS.items() if not str(motivo).strip()]
    if sem_motivo:
        raise ValueError(f"Registre o motivo da exclusão das unidades {sem_motivo} em UNIDADES_EXCLUIDAS.")
    categorias_invalidas = sorted(set(CATEGORIAS_EXCLUIDAS) - CATEGORIAS_VALIDAS)
    if categorias_invalidas:
        raise ValueError(f"CATEGORIAS_EXCLUIDAS contém categorias não previstas: {categorias_invalidas}")

    texto_titulo = documento["titulo"]
    if documento["subtitulo"]:
        # Título e subtítulo separados por quebra de linha: a quebra impede que a última
        # palavra do título e a primeira do subtítulo formem um bigrama, sem acrescentar
        # nenhuma pontuação ao texto.
        texto_titulo += "\n" + documento["subtitulo"]
    todas = [{"id": 0, "tipo": "title", "categoria": None, "texto": texto_titulo}] + documento["unidades"]

    filtros = [
        ("Título e subtítulo (unidade 0)", not INCLUIR_TITULO_NA_CONTAGEM,
         lambda u: u["id"] == 0),
        ("Títulos internos (type = heading)", not INCLUIR_TITULOS_INTERNOS,
         lambda u: u["tipo"] == "heading"),
        (f"Categorias excluídas ({', '.join(CATEGORIAS_EXCLUIDAS) or 'nenhuma'})", bool(CATEGORIAS_EXCLUIDAS),
         lambda u: u["categoria"] in CATEGORIAS_EXCLUIDAS),
        ("Unidades excluídas (UNIDADES_EXCLUIDAS)", bool(UNIDADES_EXCLUIDAS),
         lambda u: u["id"] in UNIDADES_EXCLUIDAS),
    ]
    retiradas = {nome: [] for nome, _, _ in filtros}
    mantidas = []
    for unidade in todas:
        for nome, ativo, regra in filtros:
            if ativo and regra(unidade):
                retiradas[nome].append(unidade)
                break
        else:
            mantidas.append(unidade)
    return mantidas, [(nome, ativo, retiradas[nome]) for nome, ativo, _ in filtros]


def montar_stopwords(nlp):
    """Lista de stopwords do spaCy, ajustada pelas listas editáveis da configuração."""
    base = {w.lower() for w in nlp.Defaults.stop_words}
    return (base | {w.lower() for w in STOPWORDS_ADICIONAIS}) - {w.lower() for w in STOPWORDS_MANTIDAS}


def unir_compostos_hifenizados(doc):
    """Reúne em um único token as palavras compostas com hífen (ex.: multi-stakeholder,
    human-centric, state-of-the-art), que o spaCy separa em vários tokens.
    Só une quando não há espaço em volta do hífen."""
    trechos, i, n = [], 0, len(doc)
    while i < n:
        j = i
        while (j + 2 < n and doc[j].is_alpha and doc[j].whitespace_ == ""
               and doc[j + 1].text in HIFENS and doc[j + 1].whitespace_ == ""
               and doc[j + 2].is_alpha):
            j += 2
        if j > i:
            trechos.append(doc[i:j + 1])
            i = j + 1
        else:
            i += 1
    with doc.retokenize() as rt:
        for trecho in trechos:
            ultimo = trecho[-1]
            # Lema do composto = forma escrita; se o último elemento estiver no plural,
            # usa-se o lema dele (multi-stakeholders -> multi-stakeholder).
            if ultimo.tag_ in {"NNS", "NNPS"}:
                lema = trecho.text[: len(trecho.text) - len(ultimo.text)] + ultimo.lemma_
            else:
                lema = trecho.text
            rt.merge(trecho, attrs={"LEMMA": lema, "POS": ultimo.pos_, "TAG": ultimo.tag_})
    return doc


COLUNAS_TOKENS = ["unidade", "frase", "i", "posicao", "forma", "lema", "classe", "etiqueta", "conteudo"]


def processar_documento(unidades, nlp, stopwords):
    """Aplica o pré-processamento a cada unidade (título, parágrafo ou item) e devolve uma
    tabela com uma linha por token (palavra), na ordem original do documento."""
    linhas, posicao = [], 0
    for unidade, doc in zip(unidades, nlp.pipe([u["texto"] for u in unidades])):
        doc = unir_compostos_hifenizados(doc)
        for k, frase in enumerate(doc.sents):
            for tok in frase:
                if not RE_PALAVRA.fullmatch(tok.text):
                    continue
                lema = normalizar_hifen(tok.lemma_.lower())
                lema = CORRECOES_DE_LEMA.get(lema, lema)
                linhas.append({
                    "unidade": unidade["id"],
                    "frase": f"{unidade['id']}.{k}",
                    "i": tok.i,                      # índice na unidade (detecta adjacência)
                    "posicao": posicao,              # posição no documento inteiro
                    "forma": normalizar_hifen(tok.text.lower()),
                    "lema": lema,
                    "classe": tok.pos_,
                    "etiqueta": tok.tag_,
                })
                posicao += 1
    if not linhas:
        return pd.DataFrame(columns=COLUNAS_TOKENS)
    tokens = pd.DataFrame(linhas)
    tokens["conteudo"] = (
        tokens["classe"].isin(CLASSES_DE_CONTEUDO)
        & ~tokens["lema"].isin(stopwords)
        & (tokens["lema"].str.len() >= TAMANHO_MINIMO_LEMA)
        & tokens["lema"].map(lambda lema: bool(RE_PALAVRA.fullmatch(lema)))
    )
    return tokens


def formas_com_letras_descartadas(unidades, nlp):
    """Sequências que contêm letras, mas não formam token pela definição adotada (por
    exemplo, siglas com '&', como R&D). Servem para declarar o que fica fora da contagem."""
    contagem = {}
    for doc in nlp.pipe([u["texto"] for u in unidades]):
        for tok in unir_compostos_hifenizados(doc):
            if any(c.isalpha() for c in tok.text) and not RE_PALAVRA.fullmatch(tok.text):
                contagem[tok.text] = contagem.get(tok.text, 0) + 1
    return pd.Series(contagem, dtype=int).sort_values(ascending=False, kind="stable")


def perfil_documento(tokens, unidades, rotulo):
    """Indicadores descritivos de extensão e vocabulário do texto analisado."""
    conteudo = tokens[tokens["conteudo"]]
    tam = tokens.groupby("unidade").size().reindex([u["id"] for u in unidades], fill_value=0)
    freq_cont = conteudo["lema"].value_counts()
    return pd.Series({
        "Unidades analisadas (títulos, parágrafos e itens)": len(unidades),
        "Unidades sem nenhum token": int((tam == 0).sum()),
        "Tokens (palavras)": len(tokens),
        "Formas distintas": tokens["forma"].nunique(),
        "Tokens de conteúdo": len(conteudo),
        "Tokens de conteúdo (% dos tokens)": len(conteudo) / len(tokens) * 100,
        "Lemas de conteúdo distintos": conteudo["lema"].nunique(),
        "Lemas de conteúdo com uma única ocorrência": int((freq_cont == 1).sum()),
        "Tokens por unidade: média": tam.mean(),
        "Tokens por unidade: mediana": float(tam.median()),
        "Tokens por unidade: mínimo": int(tam.min()),
        "Tokens por unidade: máximo": int(tam.max()),
    }, name=rotulo, dtype=object)


def composicao_documento(tokens, unidades):
    """Unidades e tokens do texto analisado, por tipo de unidade e por categoria estrutural.
    Descreve o recorte analisado; não é uma análise de frequências por parte do plano."""
    tam = tokens.groupby("unidade").size()
    linhas = []
    for dimensao, chave in (("Tipo de unidade", "tipo"), ("Categoria estrutural", "categoria")):
        ordem = list(dict.fromkeys(u[chave] for u in unidades))   # ordem de aparição
        for valor in ordem:
            ids = [u["id"] for u in unidades if u[chave] == valor]
            n_tok = int(tam.reindex(ids, fill_value=0).sum())
            if chave == "tipo":
                rotulo = ROTULOS_TIPO.get(valor, valor)
            else:
                rotulo = valor if valor is not None else "sem categoria (título e subtítulo)"
            linhas.append({"dimensao": dimensao, "valor": rotulo, "unidades": len(ids),
                           "tokens": n_tok, "pct_tokens": n_tok / len(tokens) * 100})
    return pd.DataFrame(linhas).set_index(["dimensao", "valor"])


def tabela_filtros(filtros, nlp, stopwords):
    """Para cada filtro do recorte: se está ativo e quantas unidades e quantos tokens retira.
    Os tokens retirados são contados com o mesmo pré-processamento do texto analisado."""
    linhas = []
    for nome, ativo, retiradas in filtros:
        n_tok = len(processar_documento(retiradas, nlp, stopwords)) if retiradas else 0
        linhas.append({"filtro": nome, "ativo": "sim" if ativo else "não",
                       "unidades_retiradas": len(retiradas), "tokens_retirados": n_tok})
    return pd.DataFrame(linhas).set_index("filtro")


def lemas_descartados_por_stopwords(tokens, stopwords):
    """Lemas de classes de conteúdo (substantivos, nomes próprios, verbos, adjetivos e
    advérbios) que ficaram fora da contagem de conteúdo por estarem na lista de stopwords,
    com as suas frequências e formas observadas (skill 02, seção 4.2)."""
    sub = tokens[tokens["classe"].isin(CLASSES_DE_CONTEUDO) & tokens["lema"].isin(stopwords)]
    if sub.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "classes", "formas_observadas"]).rename_axis("lema")
    grupos = sub.groupby("lema")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "classes": grupos["classe"].agg(lambda s: "; ".join(f"{c} ({n})" for c, n in s.value_counts().items())),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def textos_repetidos(unidades, tokens):
    """Exame de inflação (skill 02, seção 3.5): textos idênticos presentes em mais de uma
    unidade analisada, com os tokens que a repetição acrescenta além da primeira ocorrência."""
    tam = tokens.groupby("unidade").size()
    grupos = {}
    for unidade in unidades:
        grupos.setdefault(unidade["texto"], []).append(unidade)
    linhas = []
    for texto, us in grupos.items():
        if len(us) > 1:
            n_tok = int(tam.get(us[0]["id"], 0))
            linhas.append({
                "texto": texto,
                "unidades_id": ", ".join(str(u["id"]) for u in us),
                "tipos": "; ".join(dict.fromkeys(ROTULOS_TIPO.get(u["tipo"], u["tipo"]) for u in us)),
                "ocorrencias": len(us),
                "tokens_por_ocorrencia": n_tok,
                "tokens_alem_da_primeira": n_tok * (len(us) - 1),
            })
    return pd.DataFrame(linhas, columns=["texto", "unidades_id", "tipos", "ocorrencias",
                                         "tokens_por_ocorrencia", "tokens_alem_da_primeira"])


# Rótulo com palavras seguido de número no início de um título interno (ex.: "Action 12:",
# "Axis 5 –"). A skill 01 mantém esses rótulos no texto (skill 01, seção 4.3).
RE_ROTULO_NUMERADO = re.compile(r"^((?:[^\W\d_]+ ){0,2}[^\W\d_]+) \d+\s*[:.)–-]")


def rotulos_numerados_em_titulos(unidades):
    """Exame de inflação (skill 02, seção 3.5): rótulos numerados no início dos títulos
    internos. Cada título acrescenta uma ocorrência de cada palavra do rótulo."""
    achados = {}
    for unidade in unidades:
        if unidade["tipo"] != "heading":
            continue
        m = RE_ROTULO_NUMERADO.match(unidade["texto"])
        if m:
            achados.setdefault(m.group(1), []).append(unidade["id"])
    linhas = [{
        "rotulo": f"{rotulo} N",
        "titulos_internos": len(ids),
        "palavras_acrescentadas": len(ids) * len(rotulo.split()),
        "primeira_unidade": ids[0],
        "ultima_unidade": ids[-1],
    } for rotulo, ids in achados.items()]
    tab = pd.DataFrame(linhas, columns=["rotulo", "titulos_internos", "palavras_acrescentadas",
                                        "primeira_unidade", "ultima_unidade"])
    return tab.sort_values(["titulos_internos", "rotulo"], ascending=[False, True]).set_index("rotulo")


def tabela_frequencias(tokens):
    """Frequência, taxa por 1.000 tokens, alcance e dispersão (DP normalizado) de
    TODOS os lemas de conteúdo. Ordenação: frequência decrescente; empate -> ordem alfabética."""
    n_tokens = len(tokens)
    unidades = sorted(tokens["unidade"].unique())
    proporcao_unidade = (tokens.groupby("unidade").size().reindex(unidades) / n_tokens).to_numpy()
    conteudo = tokens[tokens["conteudo"]]
    cruz = pd.crosstab(conteudo["lema"], conteudo["unidade"]).reindex(columns=unidades, fill_value=0)
    freq = cruz.sum(axis=1)
    # DP de Gries (2008), normalizado por Lijffijt & Gries (2012):
    # 0 = distribuição proporcional ao tamanho das unidades; 1 = concentração máxima.
    v = cruz.div(freq, axis=0).to_numpy()
    dp = 0.5 * np.abs(v - proporcao_unidade).sum(axis=1)
    dp_norm = dp / (1 - proporcao_unidade.min()) if len(unidades) > 1 else np.full(len(freq), np.nan)
    formas = conteudo.groupby("lema")["forma"].agg(
        lambda s: "; ".join(f"{forma} ({n})" for forma, n in s.value_counts().items())
    )
    tab = pd.DataFrame({
        "frequencia": freq,
        "por_1000_tokens": freq / n_tokens * 1000,
        "unidades_com_ocorrencia": (cruz > 0).sum(axis=1),
        "pct_unidades": (cruz > 0).sum(axis=1) / len(unidades) * 100,
        "DP_norm": dp_norm,
        "formas_observadas": formas,
    })
    tab.index.name = "lema"
    tab = tab.reset_index().sort_values(["frequencia", "lema"], ascending=[False, True])
    return tab.set_index("lema")


def relatar_empates(tab, coluna, n, rotulo):
    """Informa termos empatados no ponto de corte de um recorte 'top N'."""
    if len(tab) <= n:
        return
    corte = tab[coluna].iloc[n - 1]
    fora = tab.iloc[n:]
    fora = fora[fora[coluna] == corte]
    if len(fora):
        lista = ", ".join(fora.index[:20]) + (" …" if len(fora) > 20 else "")
        print(
            f"Empate no ponto de corte: o {n}º {rotulo} tem valor {num(corte, 2)}; "
            f"outros {len(fora)} com o mesmo valor ficaram fora do recorte "
            f"(desempate por ordem alfabética): {lista}"
        )


def tabela_bigramas(tokens):
    """Pares de tokens de conteúdo ADJACENTES no texto original (mesma frase, sem
    pontuação entre eles). Não se formam pares artificiais pela remoção de stopwords."""
    a = tokens.iloc[:-1].reset_index(drop=True)
    b = tokens.iloc[1:].reset_index(drop=True)
    ok = (
        (a["frase"].to_numpy() == b["frase"].to_numpy())
        & (b["i"].to_numpy() - a["i"].to_numpy() == 1)
        & a["conteudo"].to_numpy() & b["conteudo"].to_numpy()
    )
    pares = pd.DataFrame({
        "bigrama": (a["lema"] + " " + b["lema"])[ok],
        "forma": (a["forma"] + " " + b["forma"])[ok],
        "unidade": a["unidade"][ok],
    })
    if pares.empty:
        return pd.DataFrame(columns=["frequencia", "por_1000_tokens", "unidades_com_ocorrencia",
                                     "forma_principal", "formas_observadas"]).rename_axis("bigrama")
    grupos = pares.groupby("bigrama")
    tab = pd.DataFrame({
        "frequencia": grupos.size(),
        "por_1000_tokens": grupos.size() / len(tokens) * 1000,
        "unidades_com_ocorrencia": grupos["unidade"].nunique(),
        "forma_principal": grupos["forma"].agg(lambda s: s.value_counts().index[0]),
        "formas_observadas": grupos["forma"].agg(
            lambda s: "; ".join(f"{f} ({n})" for f, n in s.value_counts().items())
        ),
    })
    tab = tab.reset_index().sort_values(["frequencia", "bigrama"], ascending=[False, True])
    return tab.set_index("bigrama")


def grafico_dispersao_posicional(ax, posicoes_por_linha, rotulos, inicios_unidades, cor):
    """Gráfico de dispersão posicional: cada traço é uma ocorrência, na posição
    relativa em que aparece no documento (0 = início; 1 = fim)."""
    if len(inicios_unidades) <= 120:
        for x in inicios_unidades:
            ax.axvline(x, color=COR_GRADE, lw=0.5, zorder=0)
    for y, posicoes in enumerate(posicoes_por_linha):
        ax.vlines(posicoes, y - 0.36, y + 0.36, color=cor, lw=0.9)
    ax.set_yticks(range(len(rotulos)), labels=rotulos)
    ax.set_ylim(len(rotulos) - 0.5, -0.5)
    ax.set_xlim(0, 1)
    ax.grid(False)
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Posição relativa no documento (0 = início; 1 = fim)")


def versoes_ambiente(nlp):
    return {
        "python": platform.python_version(),
        "spacy": spacy.__version__,
        "modelo_spacy": f"{MODELO_SPACY} {nlp.meta.get('version', '?')}",
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    }

## 9. Carregamento e verificação do insumo

O JSON da skill 01 é lido em modo somente leitura. A execução é interrompida se: o arquivo não existir; não tiver sido produzido pela skill 01 para o corpus *Planos Nacionais de IA* (`extraction.skill` e `extraction.corpus`); os identificadores não forem sequenciais; houver unidade vazia; o número de unidades divergir do registrado na extração; faltar o título; ou o idioma não for `en`. O SHA-256 do JSON é exibido para registro. Em seguida, aplicam-se os parâmetros de recorte da configuração.

In [ ]:
documento = carregar_extracao(PASTA / ARQUIVO_JSON)
unidades, filtros = aplicar_recorte(documento)

por_tipo = pd.Series([u["tipo"] for u in documento["unidades"]]).value_counts()
print(f"Título: {documento['titulo']}")
print(f"Subtítulo: {documento['subtitulo']}")
print(f"Arquivo: {documento['arquivo']}")
print(f"SHA-256: {documento['sha256']}")
print(f"Idioma: {documento['idioma']}")
print(f"Unidades no JSON: {len(documento['unidades'])} ({por_tipo.get('heading', 0)} títulos internos, "
      f"{por_tipo.get('paragraph', 0)} parágrafos, {por_tipo.get('list_item', 0)} itens de lista)")
print(f"Título e subtítulo incluídos nas contagens (unidade 0): {'sim' if INCLUIR_TITULO_NA_CONTAGEM else 'não'}")
print(f"Títulos internos incluídos: {'sim' if INCLUIR_TITULOS_INTERNOS else 'não'}")
print(f"Categorias excluídas: {', '.join(CATEGORIAS_EXCLUIDAS) if CATEGORIAS_EXCLUIDAS else 'nenhuma'}")
print("Unidades excluídas: " + (", ".join(f"{i} ({motivo})" for i, motivo in UNIDADES_EXCLUIDAS.items())
                                if UNIDADES_EXCLUIDAS else "nenhuma"))
print(f"Unidades analisadas: {len(unidades)}")

## 10. Pré-processamento e perfil do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
nlp = spacy.load(MODELO_SPACY)
STOPWORDS = montar_stopwords(nlp)
tokens = processar_documento(unidades, nlp, STOPWORDS)
N_TOKENS = len(tokens)
N_UNIDADES = tokens["unidade"].nunique()

perfil = perfil_documento(tokens, unidades, SIGLA).to_frame()
composicao = composicao_documento(tokens, unidades)
filtros_aplicados = tabela_filtros(filtros, nlp, STOPWORDS)
fora_da_definicao = formas_com_letras_descartadas(unidades, nlp)

display(Markdown("**Perfil do texto analisado**"))
exibir(perfil)
display(Markdown("**Composição do texto analisado, por tipo de unidade e por categoria estrutural**"))
exibir(composicao)
display(Markdown("**Filtros do recorte: unidades e tokens retirados**"))
exibir(filtros_aplicados)
salvar_tabela(perfil, "00_perfil_documento")
salvar_tabela(composicao, "00_composicao_documento")
salvar_tabela(filtros_aplicados, "00_filtros_do_recorte")
salvar_tabela(tokens, "00_tokens_processados", index=False)

print(f"Stopwords aplicadas: {len(STOPWORDS)} (lista do spaCy, {len(STOPWORDS_ADICIONAIS)} adicionada(s), "
      f"{len(STOPWORDS_MANTIDAS)} mantida(s))")
print(f"Correções de lema aplicadas: {CORRECOES_DE_LEMA}")
print("Sequências com letras que não formam token pela definição adotada: "
      + ("; ".join(f"{forma} ({n})" for forma, n in fora_da_definicao.items()) if len(fora_da_definicao) else "nenhuma"))

**Método e escolhas.** Cada unidade é processada com o modelo `en_core_web_sm` do spaCy. O título e o subtítulo formam a unidade 0, separados por uma quebra de linha, que impede a formação de um bigrama entre a última palavra do título e a primeira do subtítulo sem acrescentar pontuação ao texto. Palavras compostas com hífen (*high-performance*, *well-being*) são reunidas em um só token. **Token** é toda palavra formada por letras (com hífen interno ou sigla com pontos); números, pontuação e símbolos não são tokens. O total de tokens é o **denominador** de todas as taxas por 1.000 tokens. **Tokens de conteúdo** são substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy, verificada pelo **lema**, para que todas as flexões de uma palavra recebam o mesmo tratamento. A composição por tipo e por categoria e a tabela de filtros descrevem o recorte analisado: com os parâmetros-padrão, nenhum filtro retira conteúdo. A tabela de tokens processados (`00_tokens_processados.csv`, exportada para `resultados/` apenas por comando do pesquisador) permite conferir, token a token, a forma, o lema, a classe gramatical e a decisão de inclusão.

**Como ler.** O perfil descreve a extensão do documento e de seu vocabulário e serve de referência para dimensionar as frequências apresentadas a seguir. A composição mostra quanto do texto analisado está em cada tipo de unidade e em cada categoria estrutural, o que ajuda a decidir sobre os filtros do recorte (`INCLUIR_TITULOS_INTERNOS`, `CATEGORIAS_EXCLUIDAS`).

**Interpretação.** *Observação:* o texto analisado tem 586 unidades e 16.391 tokens, dos quais 9.850 (60,09%) são de conteúdo, distribuídos em 1.747 lemas distintos; 726 desses lemas aparecem uma única vez. As unidades vão de 1 a 163 tokens (mediana de 18; média de 27,97). Os parágrafos reúnem 63,82% dos tokens; os itens de lista, 29,64%; os títulos internos, 6,48%; o título e o subtítulo, 0,06%. Por categoria estrutural, o corpo do plano (`Body`) tem 64,88% dos tokens; os anexos (`Annex`), 31,62%; a apresentação (`Foreword`), 3,44%. Com os parâmetros-padrão, nenhum filtro retirou conteúdo. *Inferência:* como cerca de um terço do texto está nos anexos, as frequências das seções seguintes refletem também o vocabulário dessas listas. Manter os anexos e a apresentação (padrão) ou retirá-los é decisão do pesquisador.

**Fragilidades e ajustes possíveis.** O lematizador e o etiquetador do spaCy cometem erros, sobretudo em siglas e termos técnicos. Por isso, as tabelas trazem as formas observadas de cada lema. Correções conhecidas ficam explícitas em `CORRECOES_DE_LEMA`. A lista de sequências com letras que não formam token mostra o que fica fora de todas as contagens: o possessivo *’s* (50), que o spaCy separa da palavra (*Brazil’s* → *Brazil* + *’s*) e que não é palavra; números com letras (*1940s*, *21st*, *3D*); siglas com `&` ou `+`, entre elas *R&D* (15, das quais 2 em fim de frase, grafadas *R&D.*), *R&D&I* (2), *ST&I* (1) e *B+P* (1); e um caso de pontuação colada (*MCTI)/Brazilian*), que faz perder as duas palavras. A perda das siglas de pesquisa e desenvolvimento (18 ocorrências de *R&D*, *R&D&I* e *ST&I*) é a mais relevante para o conteúdo. A lista de *stopwords* do spaCy inclui palavras como *may*, *must* e *should*, que ficam fora da contagem de conteúdo (seção 10.1).

### 10.1. Lemas de classes de conteúdo descartados pela lista de *stopwords*

In [ ]:
descartados = lemas_descartados_por_stopwords(tokens, STOPWORDS)
salvar_tabela(descartados, "00_lemas_descartados_stopwords")
exibir(descartados)
print(f"Síntese factual: {len(descartados)} lemas de classes de conteúdo ficaram fora da contagem de conteúdo "
      f"por estarem na lista de stopwords, somando {int(descartados['frequencia'].sum())} ocorrências "
      f"({num(descartados['frequencia'].sum() / N_TOKENS * 100, 1)}% dos {N_TOKENS} tokens).")

**Método e escolhas.** São listados os tokens cuja classe gramatical é de conteúdo (substantivo, nome próprio, verbo, adjetivo ou advérbio), mas cujo lema está na lista de *stopwords* do spaCy (`STOPWORDS`, ajustada por `STOPWORDS_ADICIONAIS` e `STOPWORDS_MANTIDAS`). Esses tokens contam no total de tokens (denominador das taxas), mas não entram nas frequências de lemas de conteúdo nem nos bigramas.

**Como ler.** Cada linha é um lema descartado, com a frequência, a taxa por 1.000 tokens, as classes gramaticais atribuídas pelo spaCy e as formas escritas. A tabela permite verificar se algum termo com carga semântica no plano ficou fora da contagem.

**Interpretação.** *Observação:* 63 lemas de classes de conteúdo ficaram fora da contagem por estarem na lista de *stopwords*, com 358 ocorrências (2,2% dos tokens). Os mais frequentes são palavras de função gramatical, de ligação ou de quantidade: *also* (34), *such* (33), *various* (26), *more* (23), *have* (20), *other* (18) e *only* (16). Poucos têm carga semântica possível no plano: *call* (8; formas *calls*, *called* e *call*), *amount* (3), *top* (2), *side* (2) e *part* (1). *Inferência:* a lista de *stopwords* retira sobretudo vocabulário funcional, e a perda de conteúdo é pequena. Se o pesquisador entender que algum desses lemas deve ser contado (por exemplo, *call*, cuja forma *calls* pode designar chamadas públicas, *calls for proposals*), basta incluí-lo em `STOPWORDS_MANTIDAS`.

**Fragilidades e ajustes possíveis.** A lista de *stopwords* é genérica e não foi feita para textos de política pública. Se o pesquisador entender que algum lema da tabela deve ser contado, basta incluí-lo em `STOPWORDS_MANTIDAS`; a alteração fica registrada na configuração.

### 10.2. Exame de inflação

*Skill 02, seção 3.5: inflação é artefato, ou seja, tudo o que multiplica um termo sem que o plano o tenha escrito mais vezes no seu texto. A frequência real é resultado e permanece na análise.*

In [ ]:
repetidos = textos_repetidos(unidades, tokens)
rotulos_numerados = rotulos_numerados_em_titulos(unidades)

display(Markdown("**Textos idênticos em mais de uma unidade analisada**"))
if repetidos.empty:
    print("Nenhum texto se repete em mais de uma unidade.")
else:
    exibir(repetidos.set_index("texto"))
display(Markdown("**Rótulos numerados no início dos títulos internos**"))
if rotulos_numerados.empty:
    print("Nenhum título interno começa com rótulo numerado.")
else:
    exibir(rotulos_numerados)
salvar_tabela(repetidos, "00_inflacao_textos_repetidos", index=False)
salvar_tabela(rotulos_numerados, "00_inflacao_rotulos_numerados")

extra = int(repetidos["tokens_alem_da_primeira"].sum()) if not repetidos.empty else 0
print(f"Síntese factual: {len(repetidos)} textos aparecem em mais de uma unidade; as repetições além da primeira "
      f"ocorrência somam {extra} tokens ({num(extra / N_TOKENS * 100, 2)}% dos tokens). "
      f"{int(rotulos_numerados['titulos_internos'].sum()) if not rotulos_numerados.empty else 0} títulos internos "
      f"começam com rótulo numerado, que acrescenta "
      f"{int(rotulos_numerados['palavras_acrescentadas'].sum()) if not rotulos_numerados.empty else 0} palavras. "
      f"Tratamento aplicado: {'unidades excluídas ' + str(sorted(UNIDADES_EXCLUIDAS)) if UNIDADES_EXCLUIDAS else 'nenhum (UNIDADES_EXCLUIDAS = {})'}.")

**Método e escolhas.** O exame procura dois artefatos que multiplicariam termos: (a) textos idênticos em mais de uma unidade analisada, com os tokens que cada repetição acrescenta além da primeira ocorrência; (b) rótulos com palavras seguidos de número no início dos títulos internos (expressão regular `RE_ROTULO_NUMERADO`), que a skill 01 mantém no texto. O exame não retira nada: qualquer tratamento é feito por parâmetros da configuração (`UNIDADES_EXCLUIDAS`, com o `id` e o motivo; `INCLUIR_TITULOS_INTERNOS`; `STOPWORDS_ADICIONAIS`), e o notebook nunca altera o JSON.

**Como ler.** A primeira tabela mostra cada texto repetido, as unidades em que aparece e o peso da repetição em tokens. A segunda mostra quantos títulos internos começam com cada rótulo numerado e quantas palavras o rótulo acrescenta ao total.

**Interpretação e decisão.** *Observação:* 19 textos aparecem em mais de uma unidade, e as repetições além da primeira ocorrência somam 175 tokens (1,07% do total). São de três tipos: (a) cinco nomes de áreas que aparecem como itens de lista (unidades 131 a 137) e voltam como títulos internos (unidades 232 a 331), como *Health* e *Agriculture and livestock*; (b) 11 títulos de eixo e de programa que aparecem duas vezes (primeiras ocorrências entre as unidades 162 e 215; segundas, entre 351 e 573); (c) três frases repetidas literalmente em itens de lista (unidades 236 e 244; 356, 360 e 364; 372 e 376), que respondem por 99 dos 175 tokens. Além disso, 93 títulos internos começam com rótulo numerado: *Action N* (54), *Impact Action N* (27), *Axis N* (10) e *Annex N* (2), que acrescentam 120 palavras. Como esses rótulos contêm a palavra *Action*, os 81 títulos com *Action N* e *Impact Action N* respondem por 81 das 148 ocorrências do lema *action* (seção 12); os 27 títulos com *Impact Action N*, por 27 das 66 de *impact*.

*Decisão:* nenhum tratamento (`UNIDADES_EXCLUIDAS = {}`). Os textos repetidos e os rótulos foram escritos pelo próprio plano, em partes diferentes da sua estrutura, e não resultam de diagramação; pela skill 02 (seção 3.5), são frequência real, e não inflação. Os rótulos numerados, além disso, ficam no texto por regra da skill 01 (seção 4.3), escolha que este notebook não contraria. O efeito é declarado nas seções 12 a 14. Se o pesquisador quiser retirá-lo, há três caminhos: (1) `INCLUIR_TITULOS_INTERNOS = False`, que retira todos os 153 títulos internos (1.062 tokens, 6,48%), e não apenas os rótulos; (2) nova extração pela skill 01, sem os rótulos numerados, por comando do pesquisador; (3) `UNIDADES_EXCLUIDAS`, para retirar unidades específicas, com o motivo registrado.

**Fragilidades e ajustes possíveis.** O exame cobre repetições literais e rótulos numerados no início dos títulos. Repetições parciais (um mesmo trecho dentro de parágrafos diferentes) não são detectadas.

## 11. Extensão das unidades textuais

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
tam_unidades = tokens.groupby("unidade").size().reindex([u["id"] for u in unidades], fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 3.2))
x = np.arange(len(tam_unidades))
ax.bar(x, tam_unidades.to_numpy(), width=0.8, color=COR)
ax.axhline(tam_unidades.mean(), color=COR_TINTA_2, lw=0.8)
ax.text(len(x) - 0.5, tam_unidades.mean(), f"média = {num(tam_unidades.mean(), 1)}",
        ha="right", va="bottom", fontsize=9, color=COR_TINTA_2)
passo = max(1, int(np.ceil(len(x) / 20)))
ax.set_xticks(x[::passo], labels=[str(u) for u in tam_unidades.index[::passo]])
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_xlabel("Unidade (id do título, parágrafo ou item), na ordem do documento")
ax.set_ylabel("Tokens (palavras)")
ax.set_title(f"{SIGLA} · extensão das unidades textuais", loc="left")
ax.grid(axis="x", visible=False)
ax.yaxis.set_major_formatter(FMT_BR)
salvar_figura(fig, "01_extensao_unidades")
plt.show()

salvar_tabela(tam_unidades.rename("tokens").to_frame(), "01_extensao_unidades")
maior, menor = tam_unidades.idxmax(), tam_unidades.idxmin()
print(f"Síntese factual: {len(tam_unidades)} unidades; média de {num(tam_unidades.mean(), 1)} tokens "
      f"(desvio-padrão {num(tam_unidades.std(), 1)}). Maior unidade: id {maior} ({tam_unidades[maior]} tokens); "
      f"menor: id {menor} ({tam_unidades[menor]} tokens).")

**Método e escolhas.** Número de tokens (palavras) de cada unidade, na ordem original do documento, incluindo a unidade 0 (título e subtítulo). A linha horizontal marca a média. Nenhuma unidade é omitida.

**Como ler.** Cada barra é uma unidade: um título interno, um parágrafo ou um item de lista. Barras altas indicam unidades extensas; a sequência mostra como o texto se organiza (por exemplo, trechos de parágrafos longos alternados com séries de títulos e itens curtos).

**Interpretação.** *Observação:* as 586 unidades têm, em média, 28,0 tokens (desvio-padrão de 30,3), mas a extensão muda ao longo do documento. Até a unidade 229, parágrafos longos, vários acima de 100 tokens, alternam-se com títulos e itens curtos; a maior unidade é a 91, com 163 tokens. A partir da unidade 230, onde começam os anexos (rótulo *Annex N*, seção 10.2), nenhuma barra alcança a linha de 40 tokens. A menor unidade é a 1, com 1 token. *Inferência:* o plano combina uma parte expositiva, em parágrafos, com uma parte final em forma de lista (as ações numeradas dos anexos, com títulos e itens curtos). A média geral esconde essa diferença, e as medidas por unidade da parte final são calculadas sobre unidades muito curtas.

**Fragilidades e ajustes possíveis.** A segmentação em unidades é a da skill 01, que segue a estrutura do original. Títulos internos e itens de lista são unidades muito curtas, o que torna instáveis as medidas por unidade (alcance e dispersão); isso deve ser lembrado na leitura das seções seguintes.

## 12. Lemas de conteúdo mais frequentes

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
freq = tabela_frequencias(tokens)
salvar_tabela(freq, "02_frequencias_lemas_completa")
relatar_empates(freq, "frequencia", TOP_N_LEMAS, "lema")
top = freq.head(TOP_N_LEMAS)

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top) + 1.3))
y = np.arange(len(top))
ax.barh(y, top["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top.index)
ax.invert_yaxis()
xmax = top["frequencia"].max()
for yi, (_, linha) in zip(y, top.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top)} lemas de conteúdo mais frequentes", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, "02_lemas_mais_frequentes")
plt.show()

exibir(top[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm", "formas_observadas"]])
n_conteudo = int(tokens["conteudo"].sum())
print(f"Síntese factual: os {len(top)} lemas acima somam {int(top['frequencia'].sum())} ocorrências, "
      f"{num(top['frequencia'].sum() / n_conteudo * 100, 1)}% dos {n_conteudo} tokens de conteúdo. "
      f"O documento tem {len(freq)} lemas de conteúdo distintos (tabela completa exportada).")

**Método e escolhas.** Frequência absoluta dos *N* lemas de conteúdo mais frequentes (*N* = `TOP_N_LEMAS`). O rótulo de cada barra traz a frequência por 1.000 tokens e o **alcance** (em quantas unidades o lema aparece). A tabela acrescenta o **DP normalizado** (dispersão) e as formas observadas. Ordenação: frequência decrescente; empates resolvidos por ordem alfabética e informados acima do gráfico quando afetam o corte. A tabela completa de todos os lemas (`02_frequencias_lemas_completa.csv`) é exportada para `resultados/` apenas por comando do pesquisador.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado).

**Interpretação.** *Observação:* *ai* é o lema mais frequente, com 538 ocorrências (32,82 por 1.000 tokens), cerca de 3,6 vezes o segundo, *development* (151); aparece em 356 das 586 unidades (60,75%). Seguem *action* (148), *brazil* (117), *public* (104), *data* (94) e *use* (89). A referência ao próprio país soma *brazil* (117) e *brazilian* (82), além de *national* (80) e *country* (65). Há um conjunto ligado a tecnologia e conhecimento (*technological* 72, *technology* 69, *innovation* 65, *infrastructure* 54, *system* 54, *research* 49) e outro ligado ao setor público (*public* 104, *sector* 62). *artificial* (56) e *intelligence* (55) aparecem quase sempre juntos (bigrama *artificial intelligence*, 54; seção 14). Os 25 lemas somam 2.346 ocorrências, 23,8% dos 9.850 tokens de conteúdo; *support* (49) empata com *research* no 25º lugar e ficou fora do recorte pela ordem alfabética. Pelo exame da seção 10.2, 81 das 148 ocorrências de *action* e 27 das 66 de *impact* vêm dos rótulos numerados dos títulos dos anexos; sem esses rótulos, *action* teria 67 ocorrências e *impact*, 39. *Inferência:* o vocabulário mais frequente combina a própria tecnologia (*ai*), o desenvolvimento, a referência nacional e o setor público. A posição de *action* e de *impact* no topo depende, em boa parte, da forma de lista numerada dos anexos, e não só do texto corrido.

**Fragilidades e ajustes possíveis.** Frequência não equivale a importância política: um termo pode ser frequente por razões de estilo ou por repetição formular. Lemas genéricos, como o próprio *ai* e o nome do país, tendem a ocupar o topo em planos sobre IA. Isso é um resultado, e não um erro, e por isso esses lemas não são removidos sem decisão do pesquisador. *AI* e *artificial intelligence* não são unificados, porque isso alteraria o texto. Se o pesquisador desejar, a unificação pode ser feita por regra explícita na configuração. Neste plano, *action* e *impact* são reforçados pelos rótulos numerados dos títulos (seção 10.2), e as siglas *R&D*, *R&D&I* e *ST&I* (18 ocorrências) ficam fora da contagem (seção 10).

## 13. Distribuição dos lemas ao longo do documento

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
lemas_disp = freq.head(TOP_N_DISPERSAO).index.tolist()
inicios = (tokens.groupby("unidade")["posicao"].min() / N_TOKENS).to_numpy()
posicoes = [tokens.loc[tokens["conteudo"] & (tokens["lema"] == lema), "posicao"].to_numpy() / N_TOKENS
            for lema in lemas_disp]
rotulos = [f"{lema}  (DP = {num(freq.loc[lema, 'DP_norm'])})" for lema in lemas_disp]

fig, ax = plt.subplots(figsize=(7.5, 0.33 * len(lemas_disp) + 1.3))
grafico_dispersao_posicional(ax, posicoes, rotulos, inicios, COR)
ax.set_title(f"{SIGLA} · posição das ocorrências dos {len(lemas_disp)} lemas mais frequentes", loc="left")
salvar_figura(fig, "03_dispersao_lemas")
plt.show()

exibir(freq.head(TOP_N_DISPERSAO)[["frequencia", "unidades_com_ocorrencia", "DP_norm"]].sort_values("DP_norm"))

**Método e escolhas.** Cada traço vertical é uma ocorrência do lema, posicionada pela sua posição relativa no documento (posição do token ÷ total de tokens). As linhas verticais claras que marcam o início de cada unidade só são desenhadas quando o documento tem até 120 unidades; com as unidades deste plano, ficam omitidas para não encobrir os traços. O DP normalizado (Gries, 2008; Lijffijt & Gries, 2012) resume a distribuição: 0 indica ocorrências proporcionais ao tamanho das unidades; 1, concentração máxima.

**Como ler.** Linhas com traços espalhados indicam termos presentes em todo o texto; traços agrupados indicam termos concentrados em uma parte (por exemplo, em uma seção específica). A tabela ordena os lemas do mais espalhado ao mais concentrado.

**Interpretação.** *Observação:* *ai* é o lema mais espalhado (DP = 0,38), com traços em quase todo o documento; seguem *development* (0,61) e *brazil* (0,69). Os mais concentrados são *increase* (0,88), *impact* (0,86) e *action* (0,82). Os anexos ocupam o último terço do documento (31,62% dos tokens, seção 10), a partir da posição relativa de cerca de 0,68. No gráfico, os traços de *increase* ficam sobretudo depois dessa posição; os de *impact* formam uma faixa densa entre cerca de 0,68 e 0,79, onde estão os títulos *Impact Action N* (unidades 233 a 346); os de *action* se adensam em torno de 0,38 a 0,45 e de 0,68 em diante. *technological*, *technology*, *innovation* e *country* têm a maior parte dos traços antes dos anexos. *public* forma dois grupos densos, em torno de 0,5–0,6 e de 0,86–0,95, e *data*, um grupo por volta de 0,87–0,89. *Inferência:* parte do vocabulário mais frequente atravessa o plano inteiro (*ai*, *development*, *brazil*); outra parte se concentra em seções específicas, em especial nos anexos, cuja forma de lista de ações reforça *increase*, *impact* e *action*.

**Fragilidades e ajustes possíveis.** O DP é sensível à segmentação: com muitas unidades curtas (títulos internos e itens de lista), um lema frequente não aparece na maioria delas, e o DP tende a ser alto mesmo quando os traços se espalham por todo o documento. O gráfico mostra posições, não intensidade.

## 14. Expressões recorrentes (bigramas)

*Visualização prévia: base inicial para a construção de novos gráficos (ver nota no início do notebook).*

In [ ]:
bigramas = tabela_bigramas(tokens)
salvar_tabela(bigramas, "04_bigramas_completa")
elegiveis = bigramas[bigramas["frequencia"] >= FREQ_MINIMA_BIGRAMA]
relatar_empates(elegiveis, "frequencia", TOP_N_BIGRAMAS, "bigrama")
topb = elegiveis.head(TOP_N_BIGRAMAS)

if topb.empty:
    print(f"Nenhum bigrama atinge a frequência mínima de {FREQ_MINIMA_BIGRAMA}; gráfico não gerado.")
else:
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(topb) + 1.3))
    y = np.arange(len(topb))
    ax.barh(y, topb["frequencia"], height=0.7, color=COR)
    ax.set_yticks(y, labels=topb["forma_principal"])
    ax.invert_yaxis()
    xmax = topb["frequencia"].max()
    for yi, (_, linha) in zip(y, topb.iterrows()):
        ax.text(linha["frequencia"] + xmax * 0.015, yi,
                f"{int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
                va="center", fontsize=9, color=COR_TINTA_2)
    ax.set_xlim(0, xmax * 1.25)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.xaxis.set_major_formatter(FMT_BR)
    ax.set_xlabel("Frequência absoluta (ocorrências)")
    ax.set_title(f"{SIGLA} · {len(topb)} expressões de duas palavras mais frequentes", loc="left")
    ax.grid(axis="y", visible=False)
    salvar_figura(fig, "04_bigramas")
    plt.show()
    exibir(topb)
print(f"Síntese factual: {len(bigramas)} bigramas distintos; {len(elegiveis)} com frequência >= {FREQ_MINIMA_BIGRAMA}.")

**Método e escolhas.** Pares de tokens de conteúdo **adjacentes no texto original**, na mesma frase e sem pontuação entre eles. Pares agrupados pelo lema (ex.: *data center* reúne *data centers*). O rótulo mostra a forma escrita mais frequente. Só entram no gráfico os bigramas com frequência mínima `FREQ_MINIMA_BIGRAMA`. Como a adjacência é verificada no texto original, a remoção de *stopwords* não cria pares artificiais (em *quality of life*, não se forma *quality life*).

**Como ler.** As barras mostram as expressões que o texto mais repete, frequentemente nomes de conceitos, programas e instituições. O rótulo indica em quantas unidades a expressão aparece.

**Interpretação.** *Observação:* há 2.351 bigramas distintos, 489 com frequência igual ou maior que 2. O mais frequente é *artificial intelligence* (54, em 49 unidades). O segundo, *impact action* (36), reúne 27 ocorrências da forma singular, que corresponde aos 27 rótulos *Impact Action N* (seção 10.2), e 9 da forma plural *impact actions*. *ai* entra em 10 dos 20 bigramas: como primeiro termo em *ai research* (24), *ai solutions* (20), *ai systems* (20), *ai models* (12), *ai infrastructure* (11), *ai applications*, *ai development* e *ai technologies* (10 cada); como segundo, em *generative ai* e *qualified ai* (10 cada). O setor público aparece em *public sector* (19) e *public services* (18). *program aims* (13, em 12 unidades) repete sempre a mesma forma escrita, e *develop country* (10) reúne *developing countries* (9) e *developed countries* (1), por efeito da lematização. *Inferência:* *ai* funciona sobretudo como modificador de substantivos (pesquisa, soluções, sistemas, modelos, infraestrutura). Parte das expressões mais repetidas é estrutural ou formular: *impact action* vem sobretudo dos rótulos; *program aims* parece abrir as descrições de programas; *brazilian artificial* (10) tende a reproduzir o nome do plano (*Brazilian Artificial Intelligence Plan*). Isso deve ser levado em conta antes de tratá-las como ênfase temática.

**Fragilidades e ajustes possíveis.** Expressões com três ou mais palavras aparecem fragmentadas em bigramas sobrepostos. Expressões interrompidas por *stopwords* não são captadas como bigramas. Os títulos internos entram na contagem por padrão: um título repetido conta a sua expressão a cada repetição, e cada rótulo *Impact Action N* conta uma ocorrência de *impact action* (seção 10.2). A lematização reúne formas de sentidos diferentes (*developing* e *developed* → *develop*).

## 15. Registro de execução e síntese

Parâmetros, versões do ambiente, *hash* do insumo e lista de saídas gravadas (skill 02, seção 4.2).

In [ ]:
PARAMETROS = {
    "SIGLA": SIGLA, "ARQUIVO_JSON": ARQUIVO_JSON, "NOME_SAIDA": NOME_SAIDA, "VERSAO_SAIDA": VERSAO_SAIDA,
    "SALVAR_SAIDAS": SALVAR_SAIDAS, "SOBRESCREVER": SOBRESCREVER,
    "INCLUIR_TITULO_NA_CONTAGEM": INCLUIR_TITULO_NA_CONTAGEM, "INCLUIR_TITULOS_INTERNOS": INCLUIR_TITULOS_INTERNOS,
    "CATEGORIAS_EXCLUIDAS": CATEGORIAS_EXCLUIDAS, "UNIDADES_EXCLUIDAS": UNIDADES_EXCLUIDAS,
    "IDIOMA_ESPERADO": IDIOMA_ESPERADO, "MODELO_SPACY": MODELO_SPACY,
    "CLASSES_DE_CONTEUDO": sorted(CLASSES_DE_CONTEUDO), "TAMANHO_MINIMO_LEMA": TAMANHO_MINIMO_LEMA,
    "STOPWORDS_ADICIONAIS": STOPWORDS_ADICIONAIS, "STOPWORDS_MANTIDAS": STOPWORDS_MANTIDAS,
    "CORRECOES_DE_LEMA": CORRECOES_DE_LEMA,
    "TOP_N_LEMAS": TOP_N_LEMAS, "TOP_N_DISPERSAO": TOP_N_DISPERSAO,
    "TOP_N_BIGRAMAS": TOP_N_BIGRAMAS, "FREQ_MINIMA_BIGRAMA": FREQ_MINIMA_BIGRAMA,
    "N_BOOTSTRAP": N_BOOTSTRAP, "NIVEL_CONFIANCA": NIVEL_CONFIANCA, "SEMENTE": SEMENTE,
}
registro = {
    "notebook": f"Análise_{SIGLA}.ipynb",
    "executado_em": datetime.now().astimezone().isoformat(timespec="seconds"),
    "insumo": {"arquivo": documento["arquivo"], "sha256": documento["sha256"]},
    "unidades_analisadas": len(unidades),
    "tokens": N_TOKENS,
    "parametros": PARAMETROS,
    "ambiente": versoes_ambiente(nlp),
}
if SALVAR_SAIDAS:
    SAIDAS_GRAVADAS.append("registro_execucao.json")
    registro["saidas_gravadas"] = sorted(SAIDAS_GRAVADAS)
    (DIR_SAIDA / "registro_execucao.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
else:
    registro["saidas_gravadas"] = "nenhuma (SALVAR_SAIDAS = False)"
print(json.dumps(registro, ensure_ascii=False, indent=2, default=str))

### Síntese

- **Insumo e recorte:** `PBIA_2025_extracao.json`, com 585 unidades, mais o título e o subtítulo (unidade 0). Nenhum filtro retirou conteúdo. O texto analisado tem 16.391 tokens, 9.850 de conteúdo e 1.747 lemas de conteúdo distintos.
- **Estrutura:** até a unidade 229, parágrafos longos; a partir da unidade 230, os anexos, em títulos e itens curtos (nenhum com 40 tokens ou mais), que reúnem 31,62% dos tokens.
- **Vocabulário:** *ai* domina (538 ocorrências; 32,82 por 1.000 tokens; presente em 60,75% das unidades; DP = 0,38). Seguem *development* (151), *action* (148), *brazil* (117), *public* (104) e *data* (94).
- **Efeito da estrutura:** os rótulos numerados dos títulos dos anexos respondem por 81 ocorrências de *action* e 27 de *impact*; os textos repetidos acrescentam 175 tokens (1,07%). Não houve tratamento, porque são texto do próprio plano (seção 10.2).
- **Expressões:** *artificial intelligence* (54) e *impact action* (36, das quais 27 vêm dos rótulos); *ai* entra em 10 dos 20 bigramas mais frequentes.
- **Decisões que cabem ao pesquisador:** manter ou retirar os anexos e a apresentação (`CATEGORIAS_EXCLUIDAS`); manter ou retirar os títulos internos (`INCLUIR_TITULOS_INTERNOS`) ou só os rótulos numerados (nova extração pela skill 01); contar lemas hoje descartados como *stopwords*, como *call* (`STOPWORDS_MANTIDAS`).

## 16. Lemas de conteúdo mais frequentes sem *Brazil* e sem nomes de instituições e órgãos brasileiros (recorte de 30, ampliado nos empates)

*Visualização construída por comando do pesquisador (2026-10-05): (1) retirar da contagem as ocorrências de* Brazil*, mantendo* Brazilian*, e os nomes de instituições e órgãos brasileiros (sobretudo ministérios), mantendo as iniciativas nacionais; (2) listar os 30 lemas mais frequentes, em vez do recorte anterior, acrescentando todos os lemas empatados com o 30º. As seções 1 a 15 foram mantidas sem nenhuma alteração.*

In [ ]:
# Parâmetros desta seção, definidos por comando do pesquisador (2026-10-05). Ficam nesta célula, e não na
# configuração (seção 7), para que as células originais do notebook permaneçam inalteradas (exceção
# registrada no texto abaixo).
#
# Expressões retiradas da contagem de conteúdo, escritas como aparecem no texto do plano. A comparação é
# feita palavra a palavra, sem distinguir maiúsculas e ignorando a pontuação entre as palavras.
# Iniciativas nacionais (planos, programas, políticas, sistemas, plataformas, redes, fundos e leis) NÃO
# entram nesta lista e continuam na contagem.
EXPRESSOES_RETIRADAS = {
    "Nome do país": ["Brazil"],
    "Ministérios": [
        "Ministry of Science, Technology and Innovation", "MCTI",
        "Ministry of Management and Innovation in Public Services", "MGI",
    ],
    "Presidência, conselhos, comitês e secretarias": [
        "President of the Republic",
        "National Council for Science and Technology", "CCT",
        "Interministerial Committee for Digital Transformation", "CIT Digital",
        "Digital Government Secretariat", "SGD",
    ],
    "Outros órgãos e entidades federais": [
        "Federal Revenue of Brazil", "RFB",
        "National School of Public Administration", "Enap",
        "FNDE",
    ],
    "Empresas, agências e entidades vinculadas": [
        "Funding Authority for Studies and Projects", "Finep",
        "Brazilian Company for Industrial Research and Innovation",
        # Mesma instituição: na unidade 16, "Brazilian" vem colado a "(MCTI)/" e não forma token (seção 10).
        "Company for Industrial Research and Innovation",
        "Embrapii",
        "Federal Data Processing Service", "Serpro",
        "Social Security Technology and Information Company", "Dataprev",
        "Energy Research Company", "EPE",
        "Center for Strategic Studies and Management", "CGEE",
    ],
    "Poder Legislativo": ["National Congress"],
    "Outras instituições": ["Brazilian Academy of Sciences", "ABC"],
}


def localizar_expressoes(tokens, expressoes):
    """Localiza no texto processado as expressões indicadas e marca os seus tokens. Cada expressão é
    comparada palavra a palavra com as formas escritas (minúsculas) dos tokens de uma mesma unidade,
    em sequência; a pontuação não forma token e, por isso, não interrompe a sequência. Prevalece a
    expressão mais longa, e cada token é marcado no máximo uma vez."""
    entradas = sorted(
        ((tuple(normalizar_hifen(p.lower()) for p in RE_PALAVRA.findall(expr)), grupo, expr)
         for grupo, lista in expressoes.items() for expr in lista),
        key=lambda e: -len(e[0]),
    )
    marcados = pd.Series(False, index=tokens.index)
    achados = []
    for uid, bloco in tokens.groupby("unidade", sort=False):
        formas, indices = bloco["forma"].tolist(), bloco.index.tolist()
        livre = [True] * len(formas)
        for chave, grupo, expr in entradas:
            n, j = len(chave), 0
            while j + n <= len(formas):
                if all(livre[j + k] and formas[j + k] == chave[k] for k in range(n)):
                    idx = indices[j:j + n]
                    for k in range(n):
                        livre[j + k] = False
                    marcados.loc[idx] = True
                    achados.append({"grupo": grupo, "expressao": expr, "unidade": uid,
                                    "tokens_de_conteudo": int(tokens.loc[idx, "conteudo"].sum())})
                    j += n
                else:
                    j += 1
    return marcados, pd.DataFrame(achados, columns=["grupo", "expressao", "unidade", "tokens_de_conteudo"])


marcados, achados = localizar_expressoes(tokens, EXPRESSOES_RETIRADAS)

# Os tokens retirados saem da contagem de conteúdo (como as stopwords), mas continuam no total de tokens,
# que segue sendo o denominador das taxas por 1.000 tokens (seção 10).
tokens_sem_inst = tokens.copy()
tokens_sem_inst["conteudo"] = tokens["conteudo"] & ~marcados
freq_sem_inst = tabela_frequencias(tokens_sem_inst)

ordem = [(g, e) for g, lista in EXPRESSOES_RETIRADAS.items() for e in lista]
resumo = (achados.groupby(["grupo", "expressao"])
          .agg(ocorrencias=("unidade", "size"), tokens_de_conteudo_retirados=("tokens_de_conteudo", "sum"),
               unidades_id=("unidade", lambda s: ", ".join(map(str, sorted(set(s))))))
          .reindex(pd.MultiIndex.from_tuples(ordem, names=["grupo", "expressao"])))
resumo[["ocorrencias", "tokens_de_conteudo_retirados"]] = resumo[["ocorrencias", "tokens_de_conteudo_retirados"]].fillna(0).astype(int)
resumo["unidades_id"] = resumo["unidades_id"].fillna("—")

efeito = pd.DataFrame({
    "frequencia_secao_12": freq["frequencia"],
    "frequencia_nesta_secao": freq_sem_inst["frequencia"].reindex(freq.index, fill_value=0),
})
efeito["diferenca"] = efeito["frequencia_nesta_secao"] - efeito["frequencia_secao_12"]
efeito = (efeito[efeito["diferenca"] != 0].reset_index()
          .sort_values(["diferenca", "lema"], ascending=[True, True]).set_index("lema"))

display(Markdown("**Expressões retiradas da contagem de conteúdo**"))
exibir(resumo)
display(Markdown("**Lemas cuja frequência mudou em relação à seção 12**"))
exibir(efeito)
salvar_tabela(resumo, "07_expressoes_retiradas")
salvar_tabela(efeito, "07_efeito_da_retirada_nos_lemas")

por_grupo = resumo.groupby(level="grupo", sort=False)["tokens_de_conteudo_retirados"].sum()
n_cont_antes, n_cont_depois = int(tokens["conteudo"].sum()), int(tokens_sem_inst["conteudo"].sum())
print(f"Síntese factual: {int(resumo['ocorrencias'].sum())} ocorrências localizadas; "
      f"{n_cont_antes - n_cont_depois} tokens de conteúdo retirados "
      f"({num((n_cont_antes - n_cont_depois) / n_cont_antes * 100, 2)}% dos {n_cont_antes}), dos quais "
      + "; ".join(f"{g}: {int(v)}" for g, v in por_grupo.items())
      + f". Restam {n_cont_depois} tokens de conteúdo; o total de tokens continua {N_TOKENS}. "
      f"Expressões sem ocorrência: {', '.join(e for (_, e), n in resumo['ocorrencias'].items() if n == 0) or 'nenhuma'}.")

**Método e escolhas.** Por comando do pesquisador, saem da contagem de conteúdo: (1) todas as ocorrências de *Brazil*, enquanto *Brazilian* continua; (2) os nomes de instituições e órgãos brasileiros, por extenso e em sigla. As iniciativas nacionais continuam. O critério de separação é este:

- **Saem** os nomes próprios de órgãos e entidades: ministérios, Presidência da República, conselhos, comitês, secretarias, Receita Federal, escola de governo, empresas e agências públicas, entidades vinculadas ao governo, Congresso Nacional e Academia Brasileira de Ciências.
- **Ficam** os nomes de iniciativas: o próprio plano e os planos, programas, políticas, sistemas, plataformas, redes, fundos e leis. Por exemplo: SUS, *New Brazil Industry* (NIB), *Ecological Transition Plan*, *Applied Research Centers* (CPA), Sinapad e Cenapads, *Brazilian Open Data Portal*, INCT, *Federal Government AI Core*, IND, Obia, CNIA4I, Pnae, CadÚnico, Ater, Fies, *Direct School Money Program*, *Plano Mais Produção*, *Programa Mais Inovação*, *Brasil Mais Produtivo*, FNDCT, Funttel e LGPD.
- **Também ficam** as referências genéricas (*federal government*, *State*, *Union*, *universities*, ICTs), porque não nomeiam um órgão específico, e as instituições estrangeiras e internacionais (por exemplo, OECD e *White House AI Council*), porque o comando trata das brasileiras.

Quando as duas regras se sobrepõem, o token sai se alguma delas o alcançar. Assim, *Brazil* sai também quando faz parte do nome de uma iniciativa (5 ocorrências em *New Brazil Industry*, cujo restante continua), e *Brazilian* sai quando faz parte do nome de uma instituição (1 ocorrência, em *Brazilian Academy of Sciences*). Os nomes de iniciativas formados por nomes de instituições perdem só a parte institucional: *MCTI Intelligent Platform* mantém *Intelligent Platform*; a rede MCTI/Embrapii mantém *Network*; *Impact actions supported by Finep* mantém tudo menos *Finep*.

As expressões, escritas como no texto, ficam em `EXPRESSOES_RETIRADAS`. A função `localizar_expressoes` as procura palavra a palavra nas formas dos tokens de cada unidade, e prevalece a expressão mais longa. Os tokens localizados saem da contagem de conteúdo, como as *stopwords*, mas continuam no total de tokens (16.391), que segue sendo o denominador das taxas por 1.000 tokens; por isso, as taxas desta seção são comparáveis às da seção 12. A expressão *Brazilian Company for Industrial Research and Innovation* tem 0 ocorrência porque, na única vez em que aparece (unidade 16), *Brazilian* vem colado a "(MCTI)/" e não forma token (seção 10); a variante sem *Brazilian* cobre esse caso. Pela mesma razão, uma das cinco siglas MCTI do texto não forma token e não entra na tabela. O parâmetro e a função ficam nesta seção, e não nas seções 7 e 8, para que as células originais permaneçam inalteradas; essa exceção à centralização dos parâmetros (skill 02, seção 4.1) fica registrada aqui. Para gerar esta seção, o notebook foi executado do início em uma cópia; as células das seções 1 a 15 deste arquivo (código e saídas) não foram alteradas.

**Como ler.** A primeira tabela mostra, por grupo, cada expressão retirada, quantas vezes foi localizada, quantos tokens de conteúdo saíram e em quais unidades. A segunda mostra todos os lemas cuja frequência mudou em relação à seção 12.

**Interpretação.** *Observação:* foram localizadas 160 ocorrências, e saíram 209 tokens de conteúdo (2,12% dos 9.850). *Brazil* responde por 116 desses tokens; com a ocorrência dentro de *Federal Revenue of Brazil*, o lema *brazil* passa de 117 para 0. Os nomes de instituições e órgãos respondem por 93 tokens. Os grupos com mais tokens retirados são as empresas, agências e entidades vinculadas (33) e a Presidência, os conselhos, os comitês e as secretarias (23). Três unidades concentram 51 dos 93 tokens institucionais: a 186 (18), a 195 (18) e a 2 (15). Fora *brazil*, as reduções são pequenas: *technology* passa de 69 para 64, *innovation* de 65 para 61, *national* de 80 para 76 e *science* de 12 para 8, porque esses lemas fazem parte de nomes como *Ministry of Science, Technology and Innovation* e *National Council for Science and Technology*. *public* passa de 104 para 102, *research* de 49 para 47 e *brazilian* de 82 para 81. *Inferência:* sem contar *Brazil*, os nomes de instituições pesam pouco nas frequências do plano (93 tokens, menos de 1% dos tokens de conteúdo) e se concentram em poucos parágrafos.

**Fragilidades e ajustes possíveis.** A separação entre instituição e iniciativa é uma decisão de interpretação, e alguns casos são de fronteira:

- SUS (sistema público de saúde), Sinapad e Cenapads (sistema e centros de processamento de alto desempenho): mantidos como iniciativas;
- *Applied Research Centers*, que o próprio plano chama de iniciativas estruturantes: mantidos;
- *Federal Government AI Core*, que o plano apresenta como iniciativa, embora seja um núcleo coordenado por órgãos: mantido;
- *Union*, entidade federativa, usada em *Union Assets* e *Union real estate*: mantida;
- FNDE: retirado como órgão. O plano só traz a sigla, e a classificação usa o conhecimento sobre a entidade (autarquia federal), não o texto.

Cada caso pode mudar de lado com uma linha em `EXPRESSOES_RETIRADAS`. As 5 ocorrências de *Brazil* em *New Brazil Industry* saem pela regra (1). Para preservar o nome dessa iniciativa por inteiro, seria preciso acrescentar uma exceção a essa regra.

In [ ]:
# Recorte desta seção, definido por comando do pesquisador (2026-10-05): 30 lemas, ampliado até incluir
# todos os lemas com a mesma frequência do 30º, para que o desempate alfabético não decida quem aparece.
TOP_N_TERMOS = 30

corte = int(freq_sem_inst["frequencia"].iloc[TOP_N_TERMOS - 1])
top_termos = freq_sem_inst[freq_sem_inst["frequencia"] >= corte]
seguinte = int(freq_sem_inst["frequencia"].iloc[len(top_termos)]) if len(freq_sem_inst) > len(top_termos) else None
print(f"Ponto de corte: o {TOP_N_TERMOS}º lema tem {corte} ocorrências. "
      + (f"Outros {len(top_termos) - TOP_N_TERMOS} lema(s) têm a mesma frequência; o recorte foi ampliado para "
         f"{len(top_termos)} lemas. " if len(top_termos) > TOP_N_TERMOS else "Não há empate no corte. ")
      + (f"O lema seguinte tem {seguinte} ocorrências." if seguinte is not None else ""))

fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(top_termos) + 1.5))
y = np.arange(len(top_termos))
ax.barh(y, top_termos["frequencia"], height=0.7, color=COR)
ax.set_yticks(y, labels=top_termos.index)
ax.invert_yaxis()
xmax = top_termos["frequencia"].max()
for yi, (_, linha) in zip(y, top_termos.iterrows()):
    ax.text(linha["frequencia"] + xmax * 0.015, yi,
            f"{num(linha['por_1000_tokens'])} por 1.000 · {int(linha['unidades_com_ocorrencia'])}/{N_UNIDADES} unid.",
            va="center", fontsize=9, color=COR_TINTA_2)
ax.set_xlim(0, xmax * 1.55)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.xaxis.set_major_formatter(FMT_BR)
ax.set_xlabel("Frequência absoluta (ocorrências)")
ax.set_title(f"{SIGLA} · {len(top_termos)} lemas de conteúdo mais frequentes\n"
             "sem Brazil e sem nomes de instituições e órgãos brasileiros", loc="left")
ax.grid(axis="y", visible=False)
salvar_figura(fig, f"08_lemas_mais_frequentes_{len(top_termos)}_sem_instituicoes")
plt.show()

tabela_top = top_termos[["frequencia", "por_1000_tokens", "unidades_com_ocorrencia", "pct_unidades", "DP_norm",
                         "formas_observadas"]].copy()
tabela_top.insert(1, "frequencia_secao_12", freq["frequencia"].reindex(tabela_top.index))
exibir(tabela_top)
salvar_tabela(freq_sem_inst, "08_frequencias_lemas_sem_instituicoes_completa")

n_conteudo_sem = int(tokens_sem_inst["conteudo"].sum())
novos = [l for l in top_termos.index if l not in freq.head(TOP_N_LEMAS).index]
fora = [l for l in freq.head(TOP_N_LEMAS).index if l not in top_termos.index]
print(f"Síntese factual: os {len(top_termos)} lemas acima somam {int(top_termos['frequencia'].sum())} ocorrências, "
      f"{num(top_termos['frequencia'].sum() / n_conteudo_sem * 100, 1)}% dos {n_conteudo_sem} tokens de conteúdo "
      f"desta seção. Em relação aos {TOP_N_LEMAS} lemas da seção 12, entram: "
      + ", ".join(f"{l} ({int(top_termos.loc[l, 'frequencia'])})" for l in novos)
      + "; sai: " + (", ".join(fora) if fora else "nenhum") + ".")

**Método e escolhas.** Mesma medida e mesmo gráfico da seção 12: frequência absoluta, com a frequência por 1.000 tokens e o **alcance** no rótulo de cada barra, e o **DP normalizado** e as formas observadas na tabela. A diferença está na contagem de conteúdo, que é a desta seção, sem *Brazil* e sem nomes de instituições e órgãos brasileiros (`freq_sem_inst`), e no recorte. O recorte é de 30 lemas (`TOP_N_TERMOS`), por comando do pesquisador, e é ampliado automaticamente para incluir todos os lemas com a mesma frequência do 30º, de modo que o desempate alfabético não decida quem aparece. A célula imprime a frequência do 30º lema e a do primeiro lema fora do recorte. Dentro do recorte, lemas com a mesma frequência ficam em ordem alfabética. A tabela traz também a frequência de cada lema na seção 12 (`frequencia_secao_12`), para comparação. O parâmetro fica nesta seção, e não na configuração, pela razão registrada acima.

**Como ler.** Barras mais longas indicam lemas mais frequentes. Um lema frequente presente em muitas unidades é estruturante no texto; um lema frequente presente em poucas unidades é localizado. A distinção aparece no rótulo (alcance) e na coluna DP_norm da tabela (0 = espalhado; 1 = concentrado). A coluna `frequencia_secao_12` mostra quanto cada lema perdeu com a retirada.

**Interpretação.** *Observação:* o 30º lema tem 39 ocorrências, e um lema empata com ele. Por isso, o recorte tem 31 lemas, com *education* e *potential* (39 cada) nas duas últimas posições; o lema seguinte tem 38. Os 31 lemas somam 2.518 ocorrências, 26,1% dos 9.641 tokens de conteúdo desta seção. *ai* (538), *development* (151) e *action* (148) continuam nos três primeiros lugares. Com a saída de *brazil*, que era o 4º lema na seção 12 (117), *public* (102) e *data* (93) passam a 4º e 5º, e *brazilian* (81), a 7º. Em relação aos 25 lemas da seção 12, entram sete: *support* (49), *program* (48), *service* (45), *develop* (44), *health* (44), *education* (39) e *potential* (39). *support* já empatava com *research* no 25º lugar da seção 12. Os lemas que entram têm DP entre 0,84 e 0,92 e estão entre os mais concentrados do recorte. Dois deles nomeiam áreas de aplicação: *health* e *education*. Dentro do recorte, as maiores perdas com a retirada são de *technology* (69 → 64), *innovation* (65 → 61) e *national* (80 → 76).

*Inferência:* sem a referência ao próprio país e sem os nomes de órgãos, o topo da lista continua com *ai*, *development* e *action*, e o resto da ordem muda pouco. A ordem da seção 12, portanto, não dependia dos nomes de instituições, a não ser por *brazil*. A ampliação para 30 lemas traz vocabulário de instrumentos (*program*, *support*, *service*) e de áreas de aplicação (*health*, *education*), concentrado em partes específicas do plano, como indica o DP.

**Fragilidades e ajustes possíveis.** Ampliar o recorte não altera nenhum valor. As fragilidades da seção 12 continuam válidas, em especial a de *action* e *impact*, reforçados pelos rótulos numerados dos títulos dos anexos (seção 10.2), e a das siglas *R&D*, que ficam fora da contagem (seção 10). A classificação entre instituição e iniciativa tem casos de fronteira, listados acima. *develop* reúne *developing* e *developed*. Os sete lemas que entram no recorte têm DP alto, de modo que a presença deles depende de poucos trechos do plano.